In [1]:
import pandas as pd
import numpy as np
import json
import pickle
import shutil
from pathlib import Path

ROOT      = Path("..")
PROCESSED = ROOT / "processed"
ASSETS    = ROOT / "app" / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROCESSED / "data_processed.parquet")
df = df.sort_values("day").reset_index(drop=True)

with open(PROCESSED / "selected_features.json") as f:
    features = json.load(f)
with open(PROCESSED / "label_encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

# 1) Light table for the Overview tab (4 columns, all rows)
overview = df[["day", "isFraud", "TransactionAmt", "ProductCD"]].copy()
# ProductCD was label-encoded in notebook 03 -> restore original letters (C, H, R, S, W)
overview["ProductCD"] = encoders["ProductCD"].inverse_transform(
    overview["ProductCD"].astype(int)
)
overview.to_parquet(ASSETS / "overview.parquet", index=False)

# 2) Test sample for the Scoring tab (100 frauds + 300 legit, original index kept)
n = len(df)
test = df.iloc[int(n * 0.85):]
fraud_idx = test[test["isFraud"] == 1].sample(100, random_state=42).index
legit_idx = test[test["isFraud"] == 0].sample(300, random_state=42).index
sample = test.loc[fraud_idx.union(legit_idx), features + ["isFraud"]]
sample.to_parquet(ASSETS / "test_sample.parquet")

# 3) Copy the remaining artifacts the app needs
shutil.copy(ROOT / "models" / "xgb_fraud_model.json", ASSETS)
shutil.copy(PROCESSED / "selected_features.json", ASSETS)
shutil.copy(PROCESSED / "optimal_threshold.json", ASSETS)
shutil.copy(ROOT / "figures" / "10_profit_curve.png", ASSETS)
shutil.copytree(ROOT / "reports", ASSETS / "reports", dirs_exist_ok=True)

for p in sorted(ASSETS.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(ASSETS)}: {p.stat().st_size / 1e6:.1f} MB")

10_profit_curve.png: 0.1 MB
optimal_threshold.json: 0.0 MB
overview.parquet: 1.4 MB
reports/drift_report_window1.html: 5.1 MB
reports/drift_report_window2.html: 5.1 MB
selected_features.json: 0.0 MB
test_sample.parquet: 0.1 MB
xgb_fraud_model.json: 3.9 MB
